### Import required modules

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression, Lasso
from sklearn.metrics import (
    accuracy_score, mean_absolute_error, mean_squared_error, root_mean_squared_error, r2_score
    
)
from sklearn.preprocessing import StandardScaler

## Load all dataset using pyhon

In [3]:
train_df = pd.read_csv("./datasets/train.csv")
val_df = pd.read_csv("./datasets/val.csv")
test_df = pd.read_csv("./datasets/test.csv")

train_df.head()

,Year,dengue_total,Location,Month,monthly_avg_temperature,avg_daily_rain,avg_daily_humidity,avg_daily_soil_moisture,avg_daily_soil_temperature,avg_daily_snowfall,avg_daily_precipitation,daily_rain_density,average_humidity,avg_soil_moisture,avg_soil_temperature,avg_snowfall,avg_precipitation
0,2024,0,KALIKOT,May,10.223880,19.783870,88.951584,0.418761,10.755308,0.000000,19.783870,high,extreme,high,low,none,high
1,2022,0,PARSA,Feb,30.428612,6.673333,62.701965,0.234122,28.814114,0.000000,6.673333,moderate,high,moderate,high,none,moderate
2,2024,19,SALYAN,Sep,10.010685,0.100000,70.479485,0.319492,11.226091,0.000000,0.100000,low,high,high,low,none,low
3,2022,1,BAJHANG,Jan,-6.802581,0.000000,34.566444,0.366226,-1.123817,0.388387,0.551613,no,low,high,low,light,low
4,2022,0,KALIKOT,May,10.058961,13.132258,89.504610,0.415793,10.209129,0.000000,13.132258,high,extreme,high,low,none,high


## Use only selected features for linear regression

In [4]:
# selected columns
selected_columns = [
    "Year",
    "dengue_total",
    "Location",
    "Month",
    "monthly_avg_temperature",
    "avg_daily_rain",
    "avg_daily_humidity",
    "avg_daily_soil_moisture",
    "avg_daily_soil_temperature",
    "avg_daily_snowfall",
    "avg_daily_precipitation"
]

# dataset with only selected columns
train = train_df[selected_columns]
val = val_df[selected_columns]
test = test_df[selected_columns]

# get dummies of required data
train_loc = pd.get_dummies(train["Location"], dtype=int)
train_month = pd.get_dummies(train["Month"], dtype=int)

test_loc = pd.get_dummies(test["Location"], dtype=int)
test_month = pd.get_dummies(test["Month"], dtype=int)

val_loc = pd.get_dummies(val["Location"], dtype=int)
val_month = pd.get_dummies(val["Month"], dtype=int)

# Final dataset ready to be used for train, val and testing
train = pd.concat([train.drop(columns=["Month", "Location"]), train_loc, train_month], axis=1)
val = pd.concat([val.drop(columns=["Month", "Location"]), val_loc, val_month], axis=1)
test = pd.concat([test.drop(columns=["Month", "Location"]), test_loc, test_month], axis=1)

# display to see basic structure
print("Train Set")
display(train.head(3))
print("Val Set")
display(val.head(3))
print("Test Set")
display(test.head(3))

Train Set


,Year,dengue_total,monthly_avg_temperature,avg_daily_rain,avg_daily_humidity,avg_daily_soil_moisture,avg_daily_soil_temperature,avg_daily_snowfall,avg_daily_precipitation,ACHHAM,ARGHAKHANCHI,BAGLUNG,BAITADI,BAJHANG,BAJURA,BANKE,BARA,BARDIYA,BHAKTAPUR,BHOJPUR,DADELDHURA,DAILEKH,DANG,DARCHULA,DHADING,DHANKUTA,DHANUSA,DOLAKHA,DOLPA,DOTI,GORKHA,GULMI,HUMLA,ILAM,JAJARKOT,JHAPA,JUMLA,KAILALI,KALIKOT,KANCHANPUR,...,MUSTANG,MYAGDI,NUWAKOT,OKHALDHUNGA,PALPA,PANCHTHAR,PARBAT,PARSA,PYUTHAN,RAMECHHAP,RASUWA,RAUTAHAT,ROLPA,RUPANDEHI,SALYAN,SANKHUWASABHA,SAPTARI,SARLAHI,SINDHULI,SINDHUPALCHOK,SIRAHA,SOLUKHUMBU,SUNSARI,SURKHET,SYANGJA,TANAHU,TAPLEJUNG,UDAYAPUR,Apr,Aug,Dec,Feb,Jan,Jul,Jun,Mar,May,Nov,Oct,Sep
0,2024,0,10.223880,19.783870,88.951584,0.418761,10.755308,0.0,19.783870,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0
1,2022,0,30.428612,6.673333,62.701965,0.234122,28.814114,0.0,6.673333,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0
2,2024,19,10.010685,0.100000,70.479485,0.319492,11.226091,0.0,0.100000,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1


Val Set


,Year,dengue_total,monthly_avg_temperature,avg_daily_rain,avg_daily_humidity,avg_daily_soil_moisture,avg_daily_soil_temperature,avg_daily_snowfall,avg_daily_precipitation,ACHHAM,ARGHAKHANCHI,BAGLUNG,BAITADI,BAJHANG,BAJURA,BANKE,BARA,BARDIYA,BHAKTAPUR,BHOJPUR,DADELDHURA,DAILEKH,DANG,DARCHULA,DHADING,DHANKUTA,DHANUSA,DOLAKHA,DOLPA,DOTI,GORKHA,GULMI,HUMLA,ILAM,JAJARKOT,JHAPA,JUMLA,KAILALI,KALIKOT,KANCHANPUR,...,MUSTANG,MYAGDI,NUWAKOT,OKHALDHUNGA,PALPA,PANCHTHAR,PARBAT,PARSA,PYUTHAN,RAMECHHAP,RASUWA,RAUTAHAT,ROLPA,RUPANDEHI,SALYAN,SANKHUWASABHA,SAPTARI,SARLAHI,SINDHULI,SINDHUPALCHOK,SIRAHA,SOLUKHUMBU,SUNSARI,SURKHET,SYANGJA,TANAHU,TAPLEJUNG,UDAYAPUR,Apr,Aug,Dec,Feb,Jan,Jul,Jun,Mar,May,Nov,Oct,Sep
0,2023,2,10.692419,2.193548,62.936783,0.239148,11.303670,0.0,2.193548,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0
1,2023,0,28.076279,8.280644,83.651380,0.388021,28.989265,0.0,8.280644,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0
2,2023,1,28.192223,10.076666,84.941230,0.381857,29.020670,0.0,10.076666,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0


Test Set


,Year,dengue_total,monthly_avg_temperature,avg_daily_rain,avg_daily_humidity,avg_daily_soil_moisture,avg_daily_soil_temperature,avg_daily_snowfall,avg_daily_precipitation,ACHHAM,ARGHAKHANCHI,BAGLUNG,BAITADI,BAJHANG,BAJURA,BANKE,BARA,BARDIYA,BHAKTAPUR,BHOJPUR,DADELDHURA,DAILEKH,DANG,DARCHULA,DHADING,DHANKUTA,DHANUSA,DOLAKHA,DOLPA,DOTI,GORKHA,GULMI,HUMLA,ILAM,JAJARKOT,JHAPA,JUMLA,KAILALI,KALIKOT,KANCHANPUR,...,MUSTANG,MYAGDI,NUWAKOT,OKHALDHUNGA,PALPA,PANCHTHAR,PARBAT,PARSA,PYUTHAN,RAMECHHAP,RASUWA,RAUTAHAT,ROLPA,RUPANDEHI,SALYAN,SANKHUWASABHA,SAPTARI,SARLAHI,SINDHULI,SINDHUPALCHOK,SIRAHA,SOLUKHUMBU,SUNSARI,SURKHET,SYANGJA,TANAHU,TAPLEJUNG,UDAYAPUR,Apr,Aug,Dec,Feb,Jan,Jul,Jun,Mar,May,Nov,Oct,Sep
0,2023,0,22.582495,15.812903,85.182590,0.421109,22.287094,0.0,15.812903,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0
1,2024,55,10.086084,0.029032,65.987830,0.326807,10.154941,0.0,0.029032,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
2,2024,4,10.825872,0.512903,77.253334,0.258161,13.172967,0.0,0.512903,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0


# Extract X and Y from data

In [5]:
# Extract train and test
X_train = train.drop(columns=["dengue_total"])
y_train = train["dengue_total"]

X_val = val.drop(columns=["dengue_total"])
y_val = val["dengue_total"]

X_test = test.drop(columns=["dengue_total"])
y_test = test["dengue_total"]
